In [1]:
from sklearn.linear_model import RidgeCV
import numpy as np
from PIL import Image
import torch.nn as nn
from open_clip import create_model_and_transforms
from safetensors.torch import load_file
import torch
from tqdm import tqdm

# load cond_proj and model
cond_proj = nn.Linear(1280, 32).cuda()
cond_proj.load_state_dict(torch.load('cond_proj.pth'))
cond_proj.eval()

model, _, preprocess = create_model_and_transforms('ViT-bigG-14', pretrained=None)
state_dict = load_file('/root/BriLLM0.5/CLIP-ViT-bigG-14-laion2B-39B-b160k/CLIP-ViT-bigG-14-laion2B-39B-b160k.safetensors', device='cpu')
model.load_state_dict(state_dict, strict=False)
model = model.visual.cuda().eval()

fMRI_cls = np.load('god_fMRI_cls.npy')  # (9,768)
image_conds = []
god_images = [Image.new('RGB', (224,224), color='black') for _ in range(9)]  # dummy

for i in tqdm(range(9), desc='Ridge alignment'):
    img = god_images[i]
    inputs = preprocess(img).unsqueeze(0).to('cuda')  # single img: preprocess(img) + unsqueeze(0)
    with torch.no_grad():
        vision_out = model(inputs)  # (1,1280)
        vision_out = vision_out.mean(1) if vision_out.dim() > 2 else vision_out
        cond = cond_proj(vision_out).cpu().numpy()[0]
    image_conds.append(cond)
image_conds = np.array(image_conds)  # (9,32)

ridge = RidgeCV(alphas=[0.1,1,10,100])
ridge.fit(fMRI_cls, image_conds)
np.save('ridge_coef.npy', ridge.coef_)  # (768,32)
print(f'Best alpha: {ridge.alpha_}, R2: {ridge.score(fMRI_cls, image_conds):.4f}')

Ridge alignment: 100%|██████████| 9/9 [00:00<00:00, 18.66it/s]


Best alpha: 0.1, R2: 1.0000


In [2]:
import torch
import torch.nn as nn
from einops import rearrange
from torch.optim import Adam
from torch.nn import functional as F
import numpy as np
from tqdm import tqdm
import logging
import torch.nn.functional as F_pad

logging.basicConfig(filename='mae_log.txt', level=logging.INFO, format='%(asctime)s - %(message)s')

class ROIMAE(nn.Module):
    def __init__(self, dim=768, patch_size=16, voxels=10000):
        super().__init__()
        self.patch_embed = nn.Conv2d(1, dim, patch_size, patch_size)
        self.cls_token = nn.Parameter(torch.randn(1,1,dim))
        self.pos_embed = nn.Parameter(torch.randn(1, 197, dim))  # 197 for 224x224
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(dim, 8, batch_first=True), 6  # batch_first=True
        )
        self.decoder = nn.Linear(dim, voxels)  # reconstruct voxels from CLS

    def forward(self, x, mask=None):  # x: (b,1,H,W)
        b, c, h, w = x.shape
        # pad to 224x224 for standard ViT
        pad_h = 224 - h
        pad_w = 224 - w
        x_padded = F_pad.pad(x, (0, pad_w, 0, pad_h), mode='constant', value=0)
        patches = self.patch_embed(x_padded).flatten(2).transpose(1,2)  # (b,196,dim)
        cls = self.cls_token.expand(b, -1, -1)
        x = torch.cat([cls, patches], 1) + self.pos_embed  # (b,197,dim)
        enc = self.encoder(x)
        cls_out = enc[:,0]  # [CLS] (b,768)
        recon = self.decoder(cls_out)  # (b,voxels)
        if mask is not None:
            return recon[mask], cls_out  # masked recon + CLS
        return recon, cls_out

mae = ROIMAE(voxels=10000).cuda()
optimizer = Adam(mae.parameters(), lr=1e-4)
god_fmri = np.load('/root/autodl-tmp/god_fMRI_raw.npy')  # (9,10000)
god_fmri_2d = god_fmri.reshape(len(god_fmri), 1, 100, 100).astype(np.float32)  # (9,1,100,100)

dataloader = torch.utils.data.DataLoader(torch.from_numpy(god_fmri_2d), batch_size=1, shuffle=True)

for epoch in range(10):
    total_loss = 0
    num_batches = 0
    pbar = tqdm(dataloader, desc=f'MAE Epoch {epoch+1}/3')
    logging.info(f'Starting MAE epoch {epoch+1}')
    for x in pbar:
        x = x.to('cuda')
        # mask 75% voxels
        flat_x = x.view(1, -1)  # (1,10000)
        mask = torch.rand_like(flat_x) < 0.75  # (1,10000)
        masked_flat = flat_x.clone()
        masked_flat[mask] = 0
        masked_x = masked_flat.view(1,1,100,100)  # back to (1,1,100,100)
        recon, cls_out = mae(masked_x, mask=mask)
        loss = F.mse_loss(recon, flat_x[mask])  # masked only
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        num_batches += 1
        pbar.set_postfix({'loss': loss.item()})
        logging.info(f'MAE Batch {num_batches}: loss = {loss.item():.4f}')
    avg_loss = total_loss / num_batches
    print(f'MAE Epoch {epoch}: Avg loss = {avg_loss:.4f}')
    logging.info(f'MAE Epoch {epoch} complete: Avg loss = {avg_loss:.4f}')
torch.save(mae.state_dict(), 'fMRI_mae.pth')

# 提取CLS
mae.eval()
cls_list = []
with torch.no_grad():
    for i in range(len(god_fmri_2d)):
        x = torch.from_numpy(god_fmri_2d[i:i+1]).to('cuda')
        _, cls_out = mae(x)
        cls_list.append(cls_out.cpu().numpy())
np.save('god_fMRI_cls.npy', np.array(cls_list).squeeze())  # (9,768)
print('MAE pretrained and CLS extracted for 9 trials.')

MAE Epoch 1/3: 100%|██████████| 9/9 [00:00<00:00, 24.01it/s, loss=0.75] 


MAE Epoch 0: Avg loss = 1.0084


MAE Epoch 2/3: 100%|██████████| 9/9 [00:00<00:00, 74.35it/s, loss=0.37] 


MAE Epoch 1: Avg loss = 0.5686


MAE Epoch 3/3: 100%|██████████| 9/9 [00:00<00:00, 75.35it/s, loss=0.776]


...

MAE Epoch 7: Avg loss = 0.2357


MAE Epoch 9/3: 100%|██████████| 9/9 [00:00<00:00, 76.08it/s, loss=0.248]


MAE Epoch 8: Avg loss = 0.2369


MAE Epoch 10/3: 100%|██████████| 9/9 [00:00<00:00, 78.15it/s, loss=0.127]


MAE Epoch 9: Avg loss = 0.2382
MAE pretrained and CLS extracted for 9 trials.


In [2]:
import numpy as np
print(np.load('god_fMRI_cls.npy').shape)  # (9,768)
print(np.load('god_fMRI_cls.npy').std())  # >0.1 (learned features)

(9, 768)
0.9990371
